<a href="https://colab.research.google.com/github/fphsFischmeister/ILAE_NeuroimagingSchool/blob/master/notebooks/01_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Google Colab"/>  </a>

# Welcome to the interactive ILAE workshop on task-based activation detection.

Author: Florian Ph.S Fischmeister, Marc Berger, Radeshyam Stepponat

---
- Part 1: Preprocessing fMRI data with fMRIPrep
- Part 2: First Level of a simple motor task


In [8]:
# get some data for presentation
!rm -rf ILAE_NeuroimagingSchool
!git clone https://github.com/fphsFischmeister/ILAE_NeuroimagingSchool.git

Cloning into 'ILAE_NeuroimagingSchool'...
remote: Enumerating objects: 116, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 116 (delta 21), reused 34 (delta 7), pack-reused 66 (from 2)
Receiving objects: 100% (116/116), 127.96 MiB | 24.90 MiB/s, done.
Resolving deltas: 100% (39/39), done.


In [9]:
# get some functional motor data for presentation
!wget https://dinlab.roentgen.meduniwien.ac.at/ILAE_NeuroimagingSchool/dataset/sub-ILAEDemo001_ses-01_task-MotorHandright_run-01_space-T1w_desc-preproc_bold.nii.gz -P ILAE_NeuroimagingSchool/dataset/func/


--2026-05-04 22:27:29--  https://dinlab.roentgen.meduniwien.ac.at/ILAE_NeuroimagingSchool/dataset/sub-ILAEDemo001_ses-01_task-MotorHandright_run-01_space-T1w_desc-preproc_bold.nii.gz
Resolving dinlab.roentgen.meduniwien.ac.at (dinlab.roentgen.meduniwien.ac.at)... 149.148.226.8
Connecting to dinlab.roentgen.meduniwien.ac.at (dinlab.roentgen.meduniwien.ac.at)|149.148.226.8|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 349179751 (333M) [application/octet-stream]
Saving to: ‘ILAE_NeuroimagingSchool/dataset/func/sub-ILAEDemo001_ses-01_task-MotorHandright_run-01_space-T1w_desc-preproc_bold.nii.gz’

sub-ILAEDemo001_ses 100%[===================>] 333.00M  73.6MB/s    in 4.6s    

2026-05-04 22:27:33 (73.2 MB/s) - ‘ILAE_NeuroimagingSchool/dataset/func/sub-ILAEDemo001_ses-01_task-MotorHandright_run-01_space-T1w_desc-preproc_bold.nii.gz’ saved [349179751/349179751]



# Part 2: First Level of a simple motor task

In this notebook, we perform a first-level task-based fMRI analysis of a simple motor paradigm using Nilearn. This analysis builds on the preprocessing notebook: we assume that the raw MRI data have already been preprocessed with fMRIPrep and that the relevant outputs are available in BIDS-Derivatives format. The main inputs are the preprocessed BOLD image, the task events file, the brain mask, and selected confound regressors from fMRIPrep. 

The goal is to estimate, for one participant and one functional run, which brain regions show BOLD signal changes associated with the motor task. To do this, we fit a first-level general linear model, or GLM, which is applied voxel by voxel. The model compares the measured BOLD time series with a predicted BOLD response derived from the experimental paradigm. Events such as motor movement in this case are described in an *events.tsv* file, transformed into predicted BOLD responses by convolution with a haemodynamic response function, or HRF, and combined into a design matrix. The statistical analysis then tests whether a contrast of design-matrix columns explains a significant proportion of the fMRI signal at each brain location.



## Input for this analysis

Here we expect the input dataset to represent the output of fMRIprep.

The expected folder structure is:

````
dataset
├── anat
│   └── sub-ILAEDemo001_ses-01_run-01_desc-preproc_T1w.nii.gz
└── func
    ├── sub-ILAEDemo001_ses-01_task-MotorHandright_run-01_desc-confounds_timeseries.tsv
    ├── sub-ILAEDemo001_ses-01_task-MotorHandright_run-01_events.tsv
    ├── sub-ILAEDemo001_ses-01_task-MotorHandright_run-01_space-T1w_desc-preproc_bold.nii.gz
````
with the following file classes:
```
preproc_bold                 the 4D fMRI time series (BOLD data)
events.tsv                   timing and labels of the task conditions
confounds_timeseries.tsv     nuisance regressors, for example motion parameters
``` 

In [10]:
# validate that the file is there
!tree ILAE_NeuroimagingSchool/dataset 

ILAE_NeuroimagingSchool/dataset
├── anat
│   └── sub-ILAEDemo001_ses-01_run-01_desc-preproc_T1w.nii.gz
└── func
    ├── sub-ILAEDemo001_ses-01_task-HomeTownWalking_run-01_desc-confounds_timeseries.tsv
    ├── sub-ILAEDemo001_ses-01_task-MotorHandright_run-01_desc-confounds_timeseries.tsv
    ├── sub-ILAEDemo001_ses-01_task-MotorHandright_run-01_space-T1w_desc-preproc_bold.nii.gz
    ├── sub-ILAEDemo001_ses-01_task-ObjectNaming_run-01_desc-confounds_timeseries.tsv
    ├── sub-ILAEDemo001_ses-01_task-Phrases_run-01_desc-confounds_timeseries.tsv
    └── sub-ILAEDemo001_ses-01_task-VerbGeneration_run-01_desc-confounds_timeseries.tsv

3 directories, 7 files


## The motor task
In a simple motor task, participants are asked to perform a movement during task blocks and remain still during rest blocks. The movement may involve finger tapping, hand opening and closing, foot movement, or another clinically relevant motor action. 

In our hand it is fistclench in an interval of 30 sec. REST, 30 sec. MOTOR, ... for 5 min.

In [11]:
# load the event file and show content

import pandas as pd

task_events_data = pd.read_csv("./ILAE_NeuroimagingSchool/dataset/func/sub-ILAEDemo001_ses-01_task-MotorHandright_run-01_events.tsv", sep="\t")
events.head()

FileNotFoundError: [Errno 2] No such file or directory: './ILAE_NeuroimagingSchool/dataset/func/sub-ILAEDemo001_ses-01_task-MotorHandright_run-01_events.tsv'